# 🃏 Poker Arena — Evidência Científica #1: CFR converge ao equilíbrio de Nash (Kuhn Poker)

**Projeto:** Poker Arena (feira de ciências) · roda no **Google Colab** (Runtime → Run all). Não precisa de GPU — Kuhn é minúsculo.

**O que este notebook prova:** o algoritmo **CFR** (Counterfactual Regret Minimization) — a base dos bots famosos (Libratus, Pluribus) — **aprende sozinho a estratégia ótima** num poker pequeno o bastante para a solução ser conhecida no papel (**Kuhn Poker**).

A prova é a **exploitability caindo a ~zero**: quanto menor, mais perto do equilíbrio de Nash (impossível de explorar).

In [ ]:
# OpenSpiel (DeepMind) — framework de jogos de informacao imperfeita (Apache-2.0)
!pip install -q open_spiel

In [ ]:
import pyspiel
from open_spiel.python.algorithms import cfr, exploitability
import matplotlib.pyplot as plt

game = pyspiel.load_game('kuhn_poker')
print('Jogo:', game.get_type().long_name)
print('Jogadores:', game.num_players())

In [ ]:
# Treina CFR e mede a exploitability ao longo das iteracoes
solver = cfr.CFRSolver(game)

iters, exploit = [], []
for i in range(1, 201):
    solver.evaluate_and_update_policy()
    if i == 1 or i % 5 == 0:
        conv = exploitability.exploitability(game, solver.average_policy())
        iters.append(i)
        exploit.append(conv)
        print(f'iter {i:4d}  exploitability = {conv:.6f}')

print()
print('Exploitability final:', exploit[-1])

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(iters, exploit, marker='o', color='#c0392b')
plt.xlabel('Iteracao de CFR')
plt.ylabel('Exploitability (escala log)')
plt.title('CFR converge ao equilibrio de Nash - Kuhn Poker')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('kuhn_cfr_convergence.png', dpi=150)
plt.show()
print('Grafico salvo: kuhn_cfr_convergence.png')

## A solução conhecida do Kuhn Poker (para comparar)

O Kuhn Poker tem um **equilíbrio de Nash conhecido no papel**, parametrizado por **α ∈ [0, 1/3]** para o primeiro jogador:
- com **Valete (J)**: aposta com probabilidade **α**;
- com **Dama (Q)**: nunca aposta de cara; paga uma aposta com prob. **1/3**;
- com **Rei (K)**: aposta com probabilidade **3α** (blefe balanceado).

Abaixo, a política média aprendida pelo CFR — compare as probabilidades com a teoria.

In [ ]:
avg = solver.average_policy()
print('Information state  ->  [P(check/fold), P(bet/call)]')
for info_state, idx in sorted(avg.state_lookup.items()):
    probs = avg.action_probability_array[idx]
    print(f'  {info_state:6s}  ->  {probs.round(3)}')

## Bônus: o mesmo CFR num jogo maior — Leduc Hold'em

Leduc já tem blefe e value bet de verdade. A *mesma* técnica continua convergindo — só mais devagar. Prova de **generalização**.

In [ ]:
leduc = pyspiel.load_game('leduc_poker')
ls = cfr.CFRSolver(leduc)
it2, ex2 = [], []
for i in range(1, 101):
    ls.evaluate_and_update_policy()
    if i == 1 or i % 5 == 0:
        c = exploitability.exploitability(leduc, ls.average_policy())
        it2.append(i)
        ex2.append(c)

plt.figure(figsize=(7, 4))
plt.semilogy(it2, ex2, marker='o', color='#2980b9')
plt.xlabel('Iteracao de CFR')
plt.ylabel('Exploitability (log)')
plt.title("CFR em Leduc Hold'em")
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('leduc_cfr_convergence.png', dpi=150)
plt.show()
print('Exploitability final (Leduc):', ex2[-1])

## Para a feira

- **Gráfico 1 (Kuhn):** a exploitability cai a ~0 → CFR **provou** que chegou ao ótimo. Resultado científico nº 1, à prova de banca.
- **Tabela de política:** o bot aprendeu a blefar com o Rei e a balancear — sozinho.
- **Gráfico 2 (Leduc):** a mesma técnica generaliza para um jogo maior.

**Próximo passo:** treinar **NFSP / Deep CFR / PPO** (também no OpenSpiel) e levar os checkpoints para o backend da Poker Arena (Modo Laboratório). Treino pesado: **Colab Pro+ (A100)**; este notebook roda em CPU em poucos minutos.